# Gabriela Mistral: Wikipedia Entity and Link Analysis

This notebook runs a reusable pipeline that collects article metadata, extracts named entities with spaCy, gathers outgoing Wikipedia links, and writes CSV files for inspection or downstream graph analysis.

**Important:** the notebook retrieves current data from Wikipedia, so results may change between runs. Candidate person links and regex-extracted biographical fields are heuristics and require validation.

## 1. Setup

Install the project dependencies from the repository root before running this notebook:

```bash
pip install -r requirements.txt
python -m spacy download en_core_web_sm
```

The analysis requires internet access to query Wikipedia.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Make the project root importable when this notebook is opened from notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analyze_wikipedia import run_analysis

print("Project root:", PROJECT_ROOT)


## 2. Run the extraction pipeline

The default article is **Gabriela Mistral**. The script writes CSV outputs into the project's `data/` directory. It limits linked-page requests to 100 by default to avoid making an unexpectedly large number of requests.

In [ ]:
output_paths = run_analysis(
    title="Gabriela Mistral",
    output_dir=PROJECT_ROOT / "data",
)
output_paths


## 3. Inspect article attributes and detected entities

In [ ]:
article_attributes = pd.read_csv(output_paths["article_attributes"])
display(article_attributes.T.rename(columns={0: "Value"}))

entities = pd.read_csv(output_paths["named_entities"])
display(entities.head(20))
print(f"Unique entity spans: {len(entities)}")


## 4. Inspect people mentions and candidate link edges

A `text_mention` edge means spaCy detected a person-labelled span in the article text. An `outgoing_link` edge means the article contains a link whose title passed the simple name heuristic. Neither edge type by itself proves a biographical or social relationship.

In [ ]:
people_mentions = pd.read_csv(output_paths["people_mentions"])
link_edges = pd.read_csv(output_paths["people_link_edges"])
display(people_mentions.head(20))
display(link_edges.head(20))


## 5. Inspect linked-page attributes and outgoing links

In [ ]:
linked_attributes = pd.read_csv(output_paths["linked_people_attributes"])
outgoing_links = pd.read_csv(output_paths["outgoing_links"])

display(linked_attributes.head(20))
print(f"Outgoing Wikipedia links: {len(outgoing_links)}")
print(f"Candidate linked pages with retrieved attributes: {len(linked_attributes)}")


## 6. Data quality notes

- Review `N/A` values; they indicate that a regular expression did not match, not necessarily that the fact is unavailable.
- The name-based person filter can include non-people and miss genuine people.
- Named entity recognition can produce ambiguous or incorrect labels.
- Before using the CSVs in a research analysis, manually validate a sample and document the Wikipedia revision/date.
- For large-scale collection, set `WIKIPEDIA_USER_AGENT` to a descriptive value with a valid contact method and respect Wikimedia API guidance.